# Week 11: Neural Network Basics
# 第 11 週：神經網路基礎 -- 激活函數、正則化與批次正規化

## 學習目標 Learning Objectives
1. 理解感知器 (Perceptron) 模型，從零實作並視覺化決策邊界
2. 掌握主流激活函數 (Activation Functions) 的特性與導數
3. 理解 XOR 問題與多層網路的必要性
4. 直覺理解前向傳播 (Forward Propagation) 的矩陣運算
5. 手動計算反向傳播 (Backpropagation) 的梯度
6. 使用 sklearn MLPClassifier 比較不同架構
7. 視覺化 L1/L2 正則化與 Dropout 效果
8. 理解批次正規化 (Batch Normalization) 的原理

In [ ]:
# 匯入必要套件 Import required packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyArrowPatch, Circle
import matplotlib.patches as mpatches

from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

# 設定中文字型
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

np.random.seed(42)

print('All packages imported successfully!')

---
## Part 1: 感知器 Perceptron -- 最簡單的人工神經元

感知器 (Rosenblatt, 1958) 是最簡單的人工神經元模型：

$$z = \mathbf{w}^T \mathbf{x} + b = \sum_{i=1}^{n} w_i x_i + b$$

$$\hat{y} = \text{step}(z) = \begin{cases} 1 & \text{if } z \geq 0 \\ 0 & \text{if } z < 0 \end{cases}$$

In [ ]:
# 從零實作感知器 Perceptron from scratch
class Perceptron:
    """Simple Perceptron implementation using numpy."""
    def __init__(self, learning_rate=0.1, n_iterations=100):
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.weights = None
        self.bias = None
        self.errors_per_epoch = []

    def _step(self, z):
        return (z >= 0).astype(int)

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.errors_per_epoch = []

        for epoch in range(self.n_iter):
            errors = 0
            for xi, yi in zip(X, y):
                z = np.dot(xi, self.weights) + self.bias
                y_pred = self._step(z)
                update = self.lr * (yi - y_pred)
                self.weights += update * xi
                self.bias += update
                errors += int(update != 0.0)
            self.errors_per_epoch.append(errors)
            if errors == 0:
                break
        return self

    def predict(self, X):
        z = X @ self.weights + self.bias
        return self._step(z)

# Create linearly separable data
np.random.seed(42)
X_lin = np.vstack([
    np.random.randn(50, 2) + np.array([2, 2]),
    np.random.randn(50, 2) + np.array([-2, -2])
])
y_lin = np.array([1]*50 + [0]*50)

# Train perceptron
ptron = Perceptron(learning_rate=0.1, n_iterations=50)
ptron.fit(X_lin, y_lin)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: decision boundary
ax1 = axes[0]
xx, yy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
Z = ptron.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax1.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
ax1.scatter(X_lin[y_lin==0, 0], X_lin[y_lin==0, 1], c='blue', s=40,
            edgecolors='black', label='Class 0')
ax1.scatter(X_lin[y_lin==1, 0], X_lin[y_lin==1, 1], c='red', s=40,
            edgecolors='black', label='Class 1')
ax1.set_xlabel('$x_1$', fontsize=12)
ax1.set_ylabel('$x_2$', fontsize=12)
ax1.set_title('Perceptron Decision Boundary', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: error convergence
ax2 = axes[1]
ax2.plot(range(1, len(ptron.errors_per_epoch)+1), ptron.errors_per_epoch,
         'bo-', linewidth=2, markersize=6)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Number of Misclassifications', fontsize=12)
ax2.set_title('Perceptron Convergence', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

acc = accuracy_score(y_lin, ptron.predict(X_lin))
print(f'Perceptron accuracy: {acc:.4f}')
print(f'Weights: {ptron.weights}')
print(f'Bias: {ptron.bias:.4f}')
print(f'Converged in {len(ptron.errors_per_epoch)} epochs')

---
## Part 2: 激活函數 Activation Functions

激活函數的核心作用：**引入非線性 (Non-linearity)**。沒有激活函數，多層網路等價於單層線性變換。

| 函數 | 公式 | 輸出範圍 |
|------|------|----------|
| Sigmoid | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $(0, 1)$ |
| Tanh | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$ |
| ReLU | $\max(0, z)$ | $[0, \infty)$ |
| Leaky ReLU | $\max(\alpha z, z)$ | $(-\infty, \infty)$ |
| ELU | $z$ if $z>0$, $\alpha(e^z-1)$ if $z \leq 0$ | $(-\alpha, \infty)$ |
| Swish | $z \cdot \sigma(z)$ | $(-0.28, \infty)$ |

In [ ]:
# 定義六種激活函數及其導數
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh_fn(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1 - np.tanh(z)**2

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def leaky_relu(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_deriv(z, alpha=0.1):
    return np.where(z > 0, 1.0, alpha)

def elu(z, alpha=1.0):
    return np.where(z > 0, z, alpha * (np.exp(z) - 1))

def elu_deriv(z, alpha=1.0):
    return np.where(z > 0, 1.0, alpha * np.exp(z))

def swish(z):
    return z * sigmoid(z)

def swish_deriv(z):
    s = sigmoid(z)
    return s + z * s * (1 - s)

# 繪製激活函數及導數
z = np.linspace(-5, 5, 500)

activations = [
    ('Sigmoid', sigmoid, sigmoid_deriv, '#2196F3'),
    ('Tanh', tanh_fn, tanh_deriv, '#4CAF50'),
    ('ReLU', relu, relu_deriv, '#FF5722'),
    ('Leaky ReLU', leaky_relu, leaky_relu_deriv, '#9C27B0'),
    ('ELU', elu, elu_deriv, '#FF9800'),
    ('Swish', swish, swish_deriv, '#E91E63'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, (name, fn, fn_d, color) in enumerate(activations):
    row, col = idx // 3, idx % 3
    ax = axes[row, col]

    ax.plot(z, fn(z), color=color, linewidth=2.5, label=f'{name}(z)')
    ax.plot(z, fn_d(z), color=color, linewidth=2, linestyle='--',
            alpha=0.7, label=f"{name}'(z)")
    ax.axhline(y=0, color='gray', linewidth=0.5)
    ax.axvline(x=0, color='gray', linewidth=0.5)
    ax.set_xlabel('z', fontsize=11)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-2, 3])
    ax.set_xlim([-5, 5])

plt.suptitle('Activation Functions and Their Derivatives', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

print('Key observations:')
print('- Sigmoid/Tanh: derivatives shrink for large |z| -> vanishing gradient problem')
print('- ReLU: derivative = 1 for z>0, but 0 for z<0 (dying ReLU problem)')
print('- Leaky ReLU / ELU: solve dying ReLU by allowing small gradients for z<0')
print('- Swish: smooth, non-monotonic, used in modern architectures')

---
## Part 3: XOR 問題 -- 感知器的局限

Minsky & Papert (1969) 證明：**單層感知器無法解決 XOR 問題**，因為 XOR 是非線性可分的。

| $x_1$ | $x_2$ | XOR |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

解決方案：使用多層網路 (Multi-Layer Network)。

In [ ]:
# XOR 問題：感知器失敗 vs 兩層網路成功
np.random.seed(42)

# XOR data
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

# Try perceptron
ptron_xor = Perceptron(learning_rate=0.1, n_iterations=200)
ptron_xor.fit(X_xor, y_xor)
ptron_pred = ptron_xor.predict(X_xor)

print('=== Single Perceptron on XOR ===')
print(f'Predictions: {ptron_pred}')
print(f'True labels: {y_xor}')
print(f'Accuracy: {accuracy_score(y_xor, ptron_pred):.2f} -- FAILS!\n')

# 2-layer neural network from scratch
class TwoLayerNetwork:
    """Simple 2-layer neural network for XOR using numpy."""
    def __init__(self, hidden_size=4, lr=1.0, n_iter=5000):
        self.hidden_size = hidden_size
        self.lr = lr
        self.n_iter = n_iter
        self.losses = []

    def fit(self, X, y):
        np.random.seed(42)
        n_in = X.shape[1]
        n_out = 1
        h = self.hidden_size

        # Initialize weights
        self.W1 = np.random.randn(n_in, h) * 0.5
        self.b1 = np.zeros(h)
        self.W2 = np.random.randn(h, n_out) * 0.5
        self.b2 = np.zeros(n_out)
        self.losses = []

        y_col = y.reshape(-1, 1)
        for epoch in range(self.n_iter):
            # Forward
            z1 = X @ self.W1 + self.b1
            a1 = np.tanh(z1)  # hidden layer activation
            z2 = a1 @ self.W2 + self.b2
            a2 = sigmoid(z2)  # output activation

            # Loss (BCE)
            eps = 1e-8
            loss = -np.mean(y_col * np.log(a2 + eps) + (1 - y_col) * np.log(1 - a2 + eps))
            self.losses.append(loss)

            # Backward
            m = X.shape[0]
            dz2 = a2 - y_col                    # (m, 1)
            dW2 = (a1.T @ dz2) / m              # (h, 1)
            db2 = np.mean(dz2, axis=0)           # (1,)
            da1 = dz2 @ self.W2.T                # (m, h)
            dz1 = da1 * (1 - a1**2)              # tanh derivative
            dW1 = (X.T @ dz1) / m                # (n_in, h)
            db1 = np.mean(dz1, axis=0)           # (h,)

            # Update
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1
        return self

    def predict_proba(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = np.tanh(z1)
        z2 = a1 @ self.W2 + self.b2
        return sigmoid(z2)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int).ravel()

# Train 2-layer network
nn_xor = TwoLayerNetwork(hidden_size=4, lr=2.0, n_iter=5000)
nn_xor.fit(X_xor, y_xor)
nn_pred = nn_xor.predict(X_xor)

print('=== 2-Layer Network on XOR ===')
print(f'Predictions: {nn_pred}')
print(f'True labels: {y_xor}')
print(f'Accuracy: {accuracy_score(y_xor, nn_pred):.2f} -- SUCCEEDS!')

In [ ]:
# 視覺化 XOR 問題的決策邊界
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: XOR data points
ax1 = axes[0]
for i in range(4):
    color = 'red' if y_xor[i] == 1 else 'blue'
    marker = 'o' if y_xor[i] == 1 else 's'
    ax1.scatter(X_xor[i, 0], X_xor[i, 1], c=color, s=200,
                edgecolors='black', marker=marker, zorder=5, linewidths=2)
    ax1.annotate(f'({X_xor[i,0]},{X_xor[i,1]})={y_xor[i]}',
                xy=(X_xor[i,0], X_xor[i,1]),
                xytext=(X_xor[i,0]+0.15, X_xor[i,1]+0.1), fontsize=11)
ax1.set_title('XOR Problem\n(Not linearly separable!)', fontsize=13)
ax1.set_xlabel('$x_1$', fontsize=12)
ax1.set_ylabel('$x_2$', fontsize=12)
ax1.set_xlim([-0.5, 1.5])
ax1.set_ylim([-0.5, 1.5])
ax1.grid(True, alpha=0.3)

# Middle: decision boundary of 2-layer net
ax2 = axes[1]
xx_xor, yy_xor = np.meshgrid(np.linspace(-0.5, 1.5, 200),
                              np.linspace(-0.5, 1.5, 200))
Z_xor = nn_xor.predict_proba(np.c_[xx_xor.ravel(), yy_xor.ravel()]).reshape(xx_xor.shape)
ax2.contourf(xx_xor, yy_xor, Z_xor, levels=np.linspace(0, 1, 21),
             cmap='RdYlBu_r', alpha=0.7)
ax2.contour(xx_xor, yy_xor, Z_xor, levels=[0.5], colors='black',
            linewidths=2, linestyles='--')
for i in range(4):
    color = 'red' if y_xor[i] == 1 else 'blue'
    ax2.scatter(X_xor[i, 0], X_xor[i, 1], c=color, s=200,
                edgecolors='black', zorder=5, linewidths=2)
ax2.set_title('2-Layer Network\nDecision Boundary', fontsize=13)
ax2.set_xlabel('$x_1$', fontsize=12)
ax2.set_ylabel('$x_2$', fontsize=12)

# Right: training loss
ax3 = axes[2]
ax3.plot(nn_xor.losses, 'b-', linewidth=1.5)
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('BCE Loss', fontsize=12)
ax3.set_title('Training Loss', fontsize=13)
ax3.grid(True, alpha=0.3)
ax3.set_ylim([0, max(nn_xor.losses[:100])])

plt.tight_layout()
plt.show()

print('Single perceptron draws a straight line -> cannot separate XOR')
print('Two-layer network learns a curved decision boundary -> solves XOR!')

---
## Part 4: 前向傳播 Forward Propagation

前向傳播是神經網路計算預測值的過程。每一層的計算：

$$\mathbf{z}^{(l)} = \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$$
$$\mathbf{a}^{(l)} = f(\mathbf{z}^{(l)})$$

其中 $f$ 是激活函數。整個過程就是一連串的**矩陣乘法 + 激活函數**。

In [ ]:
# 前向傳播的逐步視覺化
np.random.seed(42)

# Define a small 3-2-1 network
W1_demo = np.array([[0.3, -0.5], [0.7, 0.2], [-0.4, 0.6]])
b1_demo = np.array([0.1, -0.2])
W2_demo = np.array([[0.8], [-0.3]])
b2_demo = np.array([0.1])

x_demo = np.array([1.0, 0.5, -0.3])

# Step-by-step forward pass
print('=== Forward Propagation Step by Step ===')
print(f'\nInput x = {x_demo}')

# Layer 1
z1_demo = x_demo @ W1_demo + b1_demo
a1_demo = np.tanh(z1_demo)
print(f'\n--- Layer 1 ---')
print(f'z1 = x @ W1 + b1 = {z1_demo}')
print(f'a1 = tanh(z1)    = {a1_demo}')

# Layer 2
z2_demo = a1_demo @ W2_demo + b2_demo
a2_demo = sigmoid(z2_demo)
print(f'\n--- Layer 2 (Output) ---')
print(f'z2 = a1 @ W2 + b2 = {z2_demo}')
print(f'a2 = sigmoid(z2)  = {a2_demo}')
print(f'\nFinal prediction: {a2_demo[0]:.4f}')

# Visualize matrix dimensions
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')

# Draw the computation flow
steps = [
    ('x\n(1x3)', 0.02, 0.5, '#E3F2FD'),
    ('W1\n(3x2)', 0.15, 0.7, '#FFF3E0'),
    ('z1\n(1x2)', 0.28, 0.5, '#E8F5E9'),
    ('tanh', 0.40, 0.5, '#FCE4EC'),
    ('a1\n(1x2)', 0.52, 0.5, '#E8F5E9'),
    ('W2\n(2x1)', 0.64, 0.7, '#FFF3E0'),
    ('z2\n(1x1)', 0.76, 0.5, '#E8F5E9'),
    ('sigmoid', 0.87, 0.5, '#FCE4EC'),
    ('output\n(1x1)', 0.97, 0.5, '#E3F2FD'),
]

for label, x, y, color in steps:
    ax.add_patch(plt.Rectangle((x-0.045, y-0.12), 0.09, 0.24,
                               facecolor=color, edgecolor='black',
                               linewidth=1.5, zorder=2))
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Arrows
for i in range(len(steps)-1):
    x1 = steps[i][1] + 0.045
    x2 = steps[i+1][1] - 0.045
    ax.annotate('', xy=(x2, 0.5), xytext=(x1, 0.5),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

ax.text(0.2, 0.15, 'z1 = x @ W1 + b1', fontsize=10, ha='center',
        style='italic', color='#666666')
ax.text(0.68, 0.15, 'z2 = a1 @ W2 + b2', fontsize=10, ha='center',
        style='italic', color='#666666')

ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0, 1)
ax.set_title('Forward Propagation: Matrix Operations Flow (3-2-1 Network)', fontsize=14)
plt.tight_layout()
plt.show()

print('Forward propagation is a chain of: linear transform -> activation -> linear transform -> activation')

---
## Part 5: 反向傳播直覺 Backpropagation Intuition

反向傳播基於**鏈式法則 (Chain Rule)**，計算每個權重對損失的貢獻：

$$\frac{\partial L}{\partial W^{(1)}} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z^{(2)}} \cdot \frac{\partial z^{(2)}}{\partial a^{(1)}} \cdot \frac{\partial a^{(1)}}{\partial z^{(1)}} \cdot \frac{\partial z^{(1)}}{\partial W^{(1)}}$$

直覺：如同工廠品管追責 -- 從產品瑕疵 (loss) 往回追溯每個工序 (layer) 的責任。

In [ ]:
# 反向傳播的手動計算：2-2-1 網路
np.random.seed(42)

print('=== Backpropagation Step by Step (2-2-1 Network) ===')
print('Input: x = [0.5, 0.8], Target: y = 1\n')

# Network setup
x_bp = np.array([0.5, 0.8])
y_bp = 1.0

W1_bp = np.array([[0.4, -0.3], [0.2, 0.5]])
b1_bp = np.array([0.1, -0.1])
W2_bp = np.array([[0.6], [-0.4]])
b2_bp = np.array([0.05])

# === Forward Pass ===
print('--- Forward Pass ---')
z1_bp = x_bp @ W1_bp + b1_bp
a1_bp = sigmoid(z1_bp)
z2_bp = a1_bp @ W2_bp + b2_bp
a2_bp = sigmoid(z2_bp)

print(f'z1 = {z1_bp}')
print(f'a1 = sigmoid(z1) = {a1_bp}')
print(f'z2 = {z2_bp}')
print(f'a2 = sigmoid(z2) = {a2_bp}')

# Loss: BCE
eps = 1e-8
loss_bp = -(y_bp * np.log(a2_bp + eps) + (1 - y_bp) * np.log(1 - a2_bp + eps))
print(f'Loss = {loss_bp[0]:.6f}')

# === Backward Pass ===
print('\n--- Backward Pass (Chain Rule) ---')

# dL/da2
dL_da2 = -(y_bp / (a2_bp + eps) - (1 - y_bp) / (1 - a2_bp + eps))
print(f'dL/da2 = {dL_da2}')

# da2/dz2 = sigmoid'(z2) = a2*(1-a2)
da2_dz2 = a2_bp * (1 - a2_bp)
dL_dz2 = dL_da2 * da2_dz2
print(f'dL/dz2 = dL/da2 * sigmoid\'(z2) = {dL_dz2}')

# dL/dW2 = a1^T * dL/dz2
dL_dW2 = a1_bp.reshape(-1, 1) @ dL_dz2.reshape(1, -1)
print(f'dL/dW2 = a1^T * dL/dz2 = \n{dL_dW2}')

# dL/da1 = dL/dz2 * W2^T
dL_da1 = dL_dz2 @ W2_bp.T
print(f'dL/da1 = dL/dz2 * W2^T = {dL_da1}')

# da1/dz1 = sigmoid'(z1)
da1_dz1 = a1_bp * (1 - a1_bp)
dL_dz1 = dL_da1 * da1_dz1
print(f'dL/dz1 = dL/da1 * sigmoid\'(z1) = {dL_dz1}')

# dL/dW1 = x^T * dL/dz1
dL_dW1 = x_bp.reshape(-1, 1) @ dL_dz1
print(f'dL/dW1 = x^T * dL/dz1 = \n{dL_dW1}')

print('\nThese gradients tell us how to update each weight to reduce the loss.')

In [ ]:
# 視覺化梯度在各層的變化（梯度消失問題）
np.random.seed(42)

def simulate_gradient_flow(n_layers, activation='sigmoid'):
    """Simulate gradient magnitude through layers."""
    gradient_mags = [1.0]  # start with gradient = 1
    for _ in range(n_layers):
        W_norm = np.random.randn() * 0.5 + 1.0  # random weight norm
        if activation == 'sigmoid':
            max_deriv = 0.25  # sigmoid max derivative
        elif activation == 'tanh':
            max_deriv = 0.65  # tanh average derivative in practice
        else:  # relu
            max_deriv = 0.5   # ~50% neurons active
        gradient_mags.append(gradient_mags[-1] * abs(W_norm) * max_deriv)
    return gradient_mags

n_layers = 10
fig, ax = plt.subplots(figsize=(12, 6))

for act, color, ls in [('sigmoid', 'blue', '-'), ('tanh', 'green', '--'), ('relu', 'red', '-')]:
    np.random.seed(42)
    grads = simulate_gradient_flow(n_layers, act)
    ax.semilogy(range(n_layers + 1), grads, f'{color}{ls}o', linewidth=2,
                markersize=8, label=f'{act.upper()}')

ax.set_xlabel('Layer (from output to input)', fontsize=12)
ax.set_ylabel('Gradient Magnitude (log scale)', fontsize=12)
ax.set_title('Gradient Flow Through Layers (Vanishing Gradient Problem)', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xticks(range(n_layers + 1))
ax.set_xticklabels([f'L{n_layers-i}' for i in range(n_layers + 1)])

plt.tight_layout()
plt.show()

print('Gradient vanishing problem:')
print('- Sigmoid: max derivative = 0.25 -> gradient shrinks exponentially through layers')
print('- Tanh: slightly better but still vanishes')
print('- ReLU: gradient = 1 for active neurons -> much better gradient flow')

---
## Part 6: MLPClassifier -- sklearn 的多層感知器

使用 sklearn 的 `MLPClassifier` 在非線性資料集上比較不同架構（1 層 vs 2 層 vs 3 層）的效果。

In [ ]:
# MLP on moons and circles datasets
np.random.seed(42)

# Generate datasets
X_moon, y_moon = make_moons(n_samples=500, noise=0.2, random_state=42)
X_circle, y_circle = make_circles(n_samples=500, noise=0.15, factor=0.5, random_state=42)

scaler_m = StandardScaler()
X_moon_s = scaler_m.fit_transform(X_moon)
scaler_c = StandardScaler()
X_circle_s = scaler_c.fit_transform(X_circle)

architectures = {
    '1 Layer (10)': (10,),
    '2 Layers (10,10)': (10, 10),
    '3 Layers (10,10,10)': (10, 10, 10),
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col_idx, (arch_name, hidden) in enumerate(architectures.items()):
    for row_idx, (X_data, y_data, ds_name, scaler_ds) in enumerate([
        (X_moon_s, y_moon, 'Moons', scaler_m),
        (X_circle_s, y_circle, 'Circles', scaler_c)
    ]):
        ax = axes[row_idx, col_idx]
        mlp = MLPClassifier(hidden_layer_sizes=hidden, max_iter=2000,
                           random_state=42, activation='relu', solver='adam')
        mlp.fit(X_data, y_data)
        acc = mlp.score(X_data, y_data)

        # Decision boundary
        xx, yy = np.meshgrid(np.linspace(X_data[:, 0].min()-0.5, X_data[:, 0].max()+0.5, 200),
                             np.linspace(X_data[:, 1].min()-0.5, X_data[:, 1].max()+0.5, 200))
        Z = mlp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

        ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
        ax.scatter(X_data[y_data==0, 0], X_data[y_data==0, 1],
                   c='blue', s=15, edgecolors='black', linewidths=0.5)
        ax.scatter(X_data[y_data==1, 0], X_data[y_data==1, 1],
                   c='red', s=15, edgecolors='black', linewidths=0.5)
        ax.set_title(f'{ds_name} | {arch_name}\nAcc={acc:.3f}', fontsize=11)
        ax.set_xlabel('$x_1$', fontsize=10)
        ax.set_ylabel('$x_2$', fontsize=10)

plt.suptitle('MLP Architectures: 1 vs 2 vs 3 Hidden Layers', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

print('Observations:')
print('- Even 1 hidden layer can learn non-linear boundaries')
print('- Deeper networks can learn more complex boundaries')
print('- For simple datasets, 1-2 layers are usually sufficient')

---
## Part 7: 正則化視覺化 Regularization Visualization

正則化控制模型複雜度，防止過擬合。兩種主要方式：

- **L1 正則化** (Lasso): $L_{\text{total}} = L_{\text{data}} + \lambda \sum |w_i|$ -- 產生稀疏權重
- **L2 正則化** (Ridge): $L_{\text{total}} = L_{\text{data}} + \lambda \sum w_i^2$ -- 權重均勻收縮

In [ ]:
# L1 vs L2 正則化的幾何直覺
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Left: L1 constraint region (diamond)
ax1 = axes[0]
theta = np.linspace(0, 2*np.pi, 500)
# L1: |w1| + |w2| <= 1 forms a diamond
w1_l1 = np.cos(theta)
w2_l1 = np.sin(theta)
# Convert to L1 diamond
w1_diamond = np.sign(np.cos(theta)) * np.abs(np.cos(theta))
w2_diamond = np.sign(np.sin(theta)) * (1 - np.abs(w1_diamond))
# Actually, parametric diamond
t = np.linspace(0, 2*np.pi, 500)
w1_d = np.cos(t)
w2_d = np.sin(t)
scale = 1.0 / (np.abs(np.cos(t)) + np.abs(np.sin(t)) + 1e-10)
w1_d *= scale
w2_d *= scale

ax1.fill(w1_d, w2_d, alpha=0.3, color='blue')
ax1.plot(w1_d, w2_d, 'b-', linewidth=2)
# Add loss contours
w1_opt, w2_opt = 1.5, 1.0
for r in [0.5, 1.0, 1.5, 2.0]:
    circle = plt.Circle((w1_opt, w2_opt), r, fill=False, color='red',
                         linestyle='--', alpha=0.5)
    ax1.add_patch(circle)
ax1.plot(w1_opt, w2_opt, 'r*', markersize=15, label='Unconstrained optimum')
ax1.plot(0, 1, 'go', markersize=12, label='L1 solution (sparse!)', zorder=5)
ax1.set_xlabel('$w_1$', fontsize=12)
ax1.set_ylabel('$w_2$', fontsize=12)
ax1.set_title('L1 Regularization (Lasso)\nConstraint = Diamond', fontsize=13)
ax1.set_xlim([-2.5, 3])
ax1.set_ylim([-2.5, 3])
ax1.set_aspect('equal')
ax1.legend(fontsize=9, loc='lower left')
ax1.axhline(y=0, color='gray', linewidth=0.5)
ax1.axvline(x=0, color='gray', linewidth=0.5)
ax1.grid(True, alpha=0.2)

# Middle: L2 constraint region (circle)
ax2 = axes[1]
circ_constraint = plt.Circle((0, 0), 1, fill=True, alpha=0.3, color='green')
ax2.add_patch(circ_constraint)
ax2.plot(np.cos(theta), np.sin(theta), 'g-', linewidth=2)
for r in [0.5, 1.0, 1.5, 2.0]:
    circle = plt.Circle((w1_opt, w2_opt), r, fill=False, color='red',
                         linestyle='--', alpha=0.5)
    ax2.add_patch(circle)
ax2.plot(w1_opt, w2_opt, 'r*', markersize=15, label='Unconstrained optimum')
# L2 solution: projection onto circle
norm = np.sqrt(w1_opt**2 + w2_opt**2)
ax2.plot(w1_opt/norm, w2_opt/norm, 'go', markersize=12,
         label='L2 solution (shrunk)', zorder=5)
ax2.set_xlabel('$w_1$', fontsize=12)
ax2.set_ylabel('$w_2$', fontsize=12)
ax2.set_title('L2 Regularization (Ridge)\nConstraint = Circle', fontsize=13)
ax2.set_xlim([-2.5, 3])
ax2.set_ylim([-2.5, 3])
ax2.set_aspect('equal')
ax2.legend(fontsize=9, loc='lower left')
ax2.axhline(y=0, color='gray', linewidth=0.5)
ax2.axvline(x=0, color='gray', linewidth=0.5)
ax2.grid(True, alpha=0.2)

# Right: regularization effect on MLP decision boundary
ax3 = axes[2]
alphas = [0.0001, 0.01, 1.0]
colors_a = ['#2196F3', '#4CAF50', '#FF5722']
X_m, y_m = make_moons(n_samples=200, noise=0.25, random_state=42)
scaler_r = StandardScaler()
X_m_s = scaler_r.fit_transform(X_m)

xx_r, yy_r = np.meshgrid(np.linspace(X_m_s[:,0].min()-0.5, X_m_s[:,0].max()+0.5, 200),
                          np.linspace(X_m_s[:,1].min()-0.5, X_m_s[:,1].max()+0.5, 200))

for alpha, color in zip(alphas, colors_a):
    mlp_r = MLPClassifier(hidden_layer_sizes=(20, 20), alpha=alpha,
                          max_iter=2000, random_state=42)
    mlp_r.fit(X_m_s, y_m)
    Z_r = mlp_r.predict_proba(np.c_[xx_r.ravel(), yy_r.ravel()])[:, 1].reshape(xx_r.shape)
    ax3.contour(xx_r, yy_r, Z_r, levels=[0.5], colors=[color], linewidths=2)

ax3.scatter(X_m_s[y_m==0, 0], X_m_s[y_m==0, 1], c='blue', s=15, edgecolors='black', linewidths=0.5)
ax3.scatter(X_m_s[y_m==1, 0], X_m_s[y_m==1, 1], c='red', s=15, edgecolors='black', linewidths=0.5)
ax3.set_title('L2 Regularization Effect\non Decision Boundary', fontsize=13)
ax3.set_xlabel('$x_1$', fontsize=12)
ax3.set_ylabel('$x_2$', fontsize=12)
legend_elements = [plt.Line2D([0],[0], color=c, linewidth=2, label=f'alpha={a}')
                   for a, c in zip(alphas, colors_a)]
ax3.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.show()

print('L1 (Lasso): diamond constraint -> solution tends to land on axes -> sparse weights')
print('L2 (Ridge): circular constraint -> solution uniformly shrunk -> small but non-zero weights')
print('Higher alpha = stronger regularization = smoother decision boundary')

---
## Part 8: 批次正規化 Batch Normalization Concept

批次正規化 (BatchNorm) 對每層的輸入做正規化，穩定訓練過程：

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$$
$$y_i = \gamma \hat{x}_i + \beta$$

其中 $\gamma$（縮放）和 $\beta$（平移）是可學習參數。

In [ ]:
# 視覺化 BatchNorm 的效果：層間分布偏移
np.random.seed(42)

# Simulate activations through layers WITH and WITHOUT BatchNorm
def forward_without_bn(x, n_layers=5):
    """Simulate forward pass without BatchNorm."""
    activations = [x]
    current = x
    for i in range(n_layers):
        W = np.random.randn(current.shape[1], current.shape[1]) * 0.5
        b = np.random.randn(current.shape[1]) * 0.3
        current = np.tanh(current @ W + b)
        activations.append(current)
    return activations

def batch_normalize(x, eps=1e-5):
    """Apply batch normalization."""
    mu = x.mean(axis=0)
    var = x.var(axis=0)
    x_hat = (x - mu) / np.sqrt(var + eps)
    return x_hat

def forward_with_bn(x, n_layers=5):
    """Simulate forward pass with BatchNorm."""
    activations = [x]
    current = x
    for i in range(n_layers):
        W = np.random.randn(current.shape[1], current.shape[1]) * 0.5
        b = np.random.randn(current.shape[1]) * 0.3
        z = current @ W + b
        z_bn = batch_normalize(z)  # Apply BN before activation
        current = np.tanh(z_bn)
        activations.append(current)
    return activations

# Generate input data
X_bn = np.random.randn(200, 10)

np.random.seed(42)
acts_no_bn = forward_without_bn(X_bn)
np.random.seed(42)
acts_with_bn = forward_with_bn(X_bn)

# Plot distributions
fig, axes = plt.subplots(2, 6, figsize=(20, 7))

for i in range(6):
    # Without BN
    ax_top = axes[0, i]
    data = acts_no_bn[i][:, 0] if i < len(acts_no_bn) else acts_no_bn[-1][:, 0]
    ax_top.hist(data, bins=30, color='#FF6B6B', alpha=0.7, edgecolor='black')
    ax_top.set_title(f'Layer {i}' if i > 0 else 'Input', fontsize=10)
    ax_top.set_xlim([-3, 3])
    if i == 0:
        ax_top.set_ylabel('Without BN', fontsize=11, fontweight='bold')

    # With BN
    ax_bot = axes[1, i]
    data_bn = acts_with_bn[i][:, 0] if i < len(acts_with_bn) else acts_with_bn[-1][:, 0]
    ax_bot.hist(data_bn, bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
    ax_bot.set_xlim([-3, 3])
    if i == 0:
        ax_bot.set_ylabel('With BN', fontsize=11, fontweight='bold')

plt.suptitle('Distribution Shift Across Layers: Without vs With BatchNorm', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print('Without BatchNorm: distributions shift and shrink through layers (internal covariate shift)')
print('With BatchNorm: distributions stay centered and normalized at each layer')
print('\nBatchNorm benefits: faster convergence, allows higher learning rates, regularization effect')

---
## Part 9: Dropout 視覺化 Dropout Visualization

Dropout 在訓練時隨機「關閉」神經元（以機率 $p$），防止共適應：

$$\tilde{h}_i = \begin{cases} 0 & \text{with probability } p \\ \frac{h_i}{1-p} & \text{with probability } 1-p \end{cases}$$

In [ ]:
# Dropout 視覺化
np.random.seed(42)

def draw_network(ax, layer_sizes, title, dropout_mask=None):
    """Draw a neural network diagram with optional dropout."""
    n_layers = len(layer_sizes)
    max_size = max(layer_sizes)
    v_spacing = 1.0 / (max_size + 1)
    h_spacing = 1.0 / (n_layers + 1)

    node_positions = []
    for l_idx, l_size in enumerate(layer_sizes):
        layer_pos = []
        x = (l_idx + 1) * h_spacing
        for n_idx in range(l_size):
            y = (n_idx + 1) * (1.0 / (l_size + 1))
            layer_pos.append((x, y))
        node_positions.append(layer_pos)

    # Draw edges
    for l_idx in range(n_layers - 1):
        for i, (x1, y1) in enumerate(node_positions[l_idx]):
            for j, (x2, y2) in enumerate(node_positions[l_idx + 1]):
                is_dropped_src = (dropout_mask is not None and
                                  l_idx < len(dropout_mask) and
                                  i < len(dropout_mask[l_idx]) and
                                  dropout_mask[l_idx][i] == 0)
                is_dropped_dst = (dropout_mask is not None and
                                  (l_idx+1) < len(dropout_mask) and
                                  j < len(dropout_mask[l_idx+1]) and
                                  dropout_mask[l_idx+1][j] == 0)
                if is_dropped_src or is_dropped_dst:
                    ax.plot([x1, x2], [y1, y2], 'gray', linewidth=0.3, alpha=0.2)
                else:
                    ax.plot([x1, x2], [y1, y2], 'black', linewidth=0.8, alpha=0.5)

    # Draw nodes
    for l_idx, layer in enumerate(node_positions):
        for n_idx, (x, y) in enumerate(layer):
            is_dropped = (dropout_mask is not None and
                         l_idx < len(dropout_mask) and
                         n_idx < len(dropout_mask[l_idx]) and
                         dropout_mask[l_idx][n_idx] == 0)
            if is_dropped:
                color = 'lightgray'
                ec = 'gray'
                marker = 'X'
            else:
                if l_idx == 0:
                    color = '#E3F2FD'
                elif l_idx == n_layers - 1:
                    color = '#FFCDD2'
                else:
                    color = '#C8E6C9'
                ec = 'black'
                marker = None

            circle = plt.Circle((x, y), 0.025, facecolor=color,
                               edgecolor=ec, linewidth=1.5, zorder=5)
            ax.add_patch(circle)
            if is_dropped:
                ax.text(x, y, 'X', ha='center', va='center',
                        fontsize=8, color='red', fontweight='bold', zorder=6)

    # Labels
    labels = ['Input'] + [f'Hidden {i+1}' for i in range(n_layers-2)] + ['Output']
    for l_idx in range(n_layers):
        x = (l_idx + 1) * h_spacing
        ax.text(x, -0.05, labels[l_idx], ha='center', fontsize=9)

    ax.set_xlim([0, 1])
    ax.set_ylim([-0.1, 1.1])
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=12, fontweight='bold')

layer_sizes = [4, 6, 6, 3]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full network
draw_network(axes[0], layer_sizes, 'Full Network\n(No Dropout)')

# Dropout sample 1
np.random.seed(42)
dropout_mask_1 = [np.ones(s) for s in layer_sizes]
for l in range(1, len(layer_sizes)-1):  # Don't drop input/output
    dropout_mask_1[l] = (np.random.rand(layer_sizes[l]) > 0.5).astype(float)
draw_network(axes[1], layer_sizes, 'Training: Dropout (p=0.5)\nSample 1',
             dropout_mask=dropout_mask_1)

# Dropout sample 2
np.random.seed(123)
dropout_mask_2 = [np.ones(s) for s in layer_sizes]
for l in range(1, len(layer_sizes)-1):
    dropout_mask_2[l] = (np.random.rand(layer_sizes[l]) > 0.5).astype(float)
draw_network(axes[2], layer_sizes, 'Training: Dropout (p=0.5)\nSample 2',
             dropout_mask=dropout_mask_2)

plt.suptitle('Dropout: Random Neuron Masking During Training', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Dropout key ideas:')
print('1. Each training step uses a different sub-network (ensemble effect)')
print('2. Prevents co-adaptation: neurons must learn useful features independently')
print('3. At inference: all neurons active, weights scaled by (1-p)')

In [ ]:
# Dropout 效果比較：有/無 Dropout 對過擬合的影響
np.random.seed(42)

# Generate noisy data
X_drop, y_drop = make_moons(n_samples=150, noise=0.3, random_state=42)
scaler_drop = StandardScaler()
X_drop_s = scaler_drop.fit_transform(X_drop)
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_drop_s, y_drop, test_size=0.4, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# sklearn MLPClassifier does not have dropout, but we can simulate
# the effect via different alpha (L2 reg) values
for idx, (alpha, title) in enumerate([(0.0001, 'Weak Regularization\n(alpha=0.0001, like no dropout)'),
                                       (1.0, 'Strong Regularization\n(alpha=1.0, similar to dropout effect)')]):
    ax = axes[idx]
    mlp_d = MLPClassifier(hidden_layer_sizes=(50, 50), alpha=alpha,
                          max_iter=2000, random_state=42)
    mlp_d.fit(X_tr_d, y_tr_d)
    train_acc = mlp_d.score(X_tr_d, y_tr_d)
    test_acc = mlp_d.score(X_te_d, y_te_d)

    xx_d, yy_d = np.meshgrid(
        np.linspace(X_drop_s[:,0].min()-0.5, X_drop_s[:,0].max()+0.5, 200),
        np.linspace(X_drop_s[:,1].min()-0.5, X_drop_s[:,1].max()+0.5, 200))
    Z_d = mlp_d.predict(np.c_[xx_d.ravel(), yy_d.ravel()]).reshape(xx_d.shape)

    ax.contourf(xx_d, yy_d, Z_d, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X_tr_d[y_tr_d==0, 0], X_tr_d[y_tr_d==0, 1], c='blue', s=30,
               edgecolors='black', linewidths=0.5, label='Train class 0')
    ax.scatter(X_tr_d[y_tr_d==1, 0], X_tr_d[y_tr_d==1, 1], c='red', s=30,
               edgecolors='black', linewidths=0.5, label='Train class 1')
    ax.scatter(X_te_d[y_te_d==0, 0], X_te_d[y_te_d==0, 1], c='blue', s=30,
               marker='^', edgecolors='black', linewidths=0.5, alpha=0.5)
    ax.scatter(X_te_d[y_te_d==1, 0], X_te_d[y_te_d==1, 1], c='red', s=30,
               marker='^', edgecolors='black', linewidths=0.5, alpha=0.5)
    ax.set_title(f'{title}\nTrain Acc={train_acc:.3f}, Test Acc={test_acc:.3f}', fontsize=12)
    ax.set_xlabel('$x_1$', fontsize=11)
    ax.set_ylabel('$x_2$', fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle('Regularization Effect: Weak vs Strong (Overfitting Control)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('Weak regularization: complex boundary, fits noise -> overfitting')
print('Strong regularization: smoother boundary, better generalization')
print('Dropout has a similar effect to L2 regularization in controlling overfitting')

---
## Part 10: 練習題 Exercises

完成以下練習來鞏固本週所學。

### Exercise 1: Implement a 2-layer network for AND/OR gates

使用 numpy 實作一個 2 層神經網路（類似 Part 3 的 `TwoLayerNetwork`），
分別在 AND 和 OR 邏輯閘上訓練。比較所需的訓練輪數。

| AND | OR |
|:---:|:---:|
| (0,0)->0, (0,1)->0 | (0,0)->0, (0,1)->1 |
| (1,0)->0, (1,1)->1 | (1,0)->1, (1,1)->1 |

In [ ]:
# Exercise 1: 2-layer network for AND/OR
# TODO: Define AND and OR datasets
# TODO: Train TwoLayerNetwork on each
# TODO: Print predictions and plot training loss curves

# X_logic = np.array([[0,0],[0,1],[1,0],[1,1]])
# y_and = np.array([0, 0, 0, 1])
# y_or  = np.array([0, 1, 1, 1])
#
# nn_and = TwoLayerNetwork(hidden_size=4, lr=2.0, n_iter=3000)
# nn_and.fit(X_logic, y_and)
# print('AND predictions:', nn_and.predict(X_logic))
#
# nn_or = TwoLayerNetwork(hidden_size=4, lr=2.0, n_iter=3000)
# nn_or.fit(X_logic, y_or)
# print('OR predictions:', nn_or.predict(X_logic))

### Exercise 2: Experiment with activation functions in MLPClassifier

使用 `MLPClassifier` 在 make_moons 資料集上比較不同激活函數的效果：
- `activation='relu'`
- `activation='tanh'`
- `activation='logistic'` (sigmoid)

繪製每種激活函數的決策邊界，並比較準確率。

In [ ]:
# Exercise 2: Compare activation functions in MLPClassifier
# TODO: Train MLPClassifier with relu, tanh, logistic on moons data
# TODO: Plot decision boundaries side by side

# X_ex, y_ex = make_moons(n_samples=300, noise=0.2, random_state=42)
# scaler_ex = StandardScaler()
# X_ex_s = scaler_ex.fit_transform(X_ex)
#
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
# for idx, act in enumerate(['relu', 'tanh', 'logistic']):
#     mlp = MLPClassifier(hidden_layer_sizes=(20, 20), activation=act,
#                         max_iter=2000, random_state=42)
#     mlp.fit(X_ex_s, y_ex)
#     # Plot decision boundary on axes[idx]
#     # ...

### Exercise 3: Visualize hidden layer width effect on decision boundary

使用 `MLPClassifier` 在 make_circles 資料集上比較不同隱藏層寬度的效果：
- `(2,)` -- 2 neurons
- `(10,)` -- 10 neurons
- `(50,)` -- 50 neurons
- `(200,)` -- 200 neurons

觀察：隨著寬度增加，決策邊界如何變化？是否出現過擬合？

In [ ]:
# Exercise 3: Hidden layer width vs decision boundary
# TODO: Train MLPClassifier with widths [2, 10, 50, 200] on circles data
# TODO: Plot decision boundaries in a 1x4 grid

# X_circ, y_circ = make_circles(n_samples=300, noise=0.15, factor=0.5, random_state=42)
# scaler_circ = StandardScaler()
# X_circ_s = scaler_circ.fit_transform(X_circ)
#
# widths = [2, 10, 50, 200]
# fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
# for idx, w in enumerate(widths):
#     mlp = MLPClassifier(hidden_layer_sizes=(w,), max_iter=2000, random_state=42)
#     mlp.fit(X_circ_s, y_circ)
#     # Plot decision boundary ...
#     # axes[idx].set_title(f'Width={w}, Acc={mlp.score(X_circ_s, y_circ):.3f}')

---
## 本週實作總結 Lab Summary

在這次實作中，我們完成了：

1. **感知器 (Perceptron)**：從零實作，視覺化決策邊界與收斂過程
2. **激活函數**：繪製 Sigmoid, Tanh, ReLU, Leaky ReLU, ELU, Swish 及其導數
3. **XOR 問題**：證明單層感知器失敗，2 層網路成功解決
4. **前向傳播**：逐步展示矩陣運算流程
5. **反向傳播直覺**：手動計算 2-2-1 網路的梯度，視覺化梯度消失問題
6. **MLPClassifier**：比較 1/2/3 層架構在 moons/circles 上的決策邊界
7. **正則化視覺化**：L1 vs L2 的幾何直覺，正則化強度對邊界的影響
8. **BatchNorm**：觀察各層分布偏移，以及 BN 如何穩定分布
9. **Dropout**：視覺化隨機神經元遮蔽，理解其正則化效果

### 關鍵收穫 Key Takeaways
- 激活函數引入非線性，沒有它多層網路就等於單層線性模型
- 反向傳播 = 鏈式法則的系統性應用
- ReLU 系列解決梯度消失問題，是目前隱藏層的首選
- 正則化 (L2, Dropout) 防止過擬合，BatchNorm 加速訓練

### 下週預告 Next Week Preview
**第 12 週：CNN 視覺化** -- 卷積運算、特徵圖、池化、CAM/Grad-CAM